# YgoMaster-PackBuilder — Stable ID Registry Tests

Test offline per il registry degli ID YgoMaster stabili.

Verifica:

- registry vuoto quando il file non esiste;
- load/save round-trip;
- allocazione deterministica;
- persistenza degli ID esistenti;
- nessun riciclo automatico degli ID;
- gestione di `next_pack_id`;
- rifiuto di ID duplicati o fuori range;
- rifiuto di pack key duplicate nel progetto;
- esaurimento del range custom;
- derivazione dello `shop_id`;
- stato iniziale versionato in `state/ygomaster_ids.json`.


## 1. Repository root

In [1]:
from pathlib import Path

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError("Impossibile individuare la root della repository.")

print("Repository root:", repo_root)

Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder


## 2. Import

In [2]:
from copy import deepcopy
from tempfile import TemporaryDirectory

from ygomaster_packbuilder.domain import (
    PackDefinition,
    PackType,
    ProjectDefinition,
    SourcePackRef,
    SourceProvider,
)

from ygomaster_packbuilder.registry import (
    YgoMasterIdRegistry,
    ensure_project_pack_ids,
    get_pack_shop_id,
    load_id_registry,
    save_id_registry,
)

from ygomaster_packbuilder.registry.ids import (
    PACK_ID_MIN,
    PACK_ID_MAX,
    PACK_SHOP_BASE_ID,
    REGISTRY_VERSION,
)

print("Imports OK")

Imports OK


## 3. Helper per un progetto minimale

In [3]:
def make_pack(key):
    return PackDefinition(
        key=key,
        name=key,
        description="",
        type=PackType.PERSONAL_MAIN,
        source=SourcePackRef(
            provider=SourceProvider.YGOPRODECK,
            external_id=key,
        ),
        expected_card_count=100,
        booster_size=7,
        opening_policy_key="configuration-a",
        cards=[],
    )


def make_project(*keys):
    return ProjectDefinition(
        name="Registry Test",
        packs=[
            make_pack(key)
            for key in keys
        ],
    )


print("Helpers ready")

Helpers ready


## 4. File mancante → registry vuoto

In [4]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "missing.json"

    registry = load_id_registry(path)

    assert registry.version == REGISTRY_VERSION
    assert registry.next_pack_id == PACK_ID_MIN
    assert registry.pack_ids == {}

print("Missing registry creates empty state")

Missing registry creates empty state


## 5. Stato iniziale della repository

In [5]:
state_path = (
    repo_root
    / "state"
    / "ygomaster_ids.json"
)

assert state_path.exists(), (
    "state/ygomaster_ids.json non esiste. "
    "Assicurati che non sia escluso dal .gitignore."
)

repo_registry = load_id_registry(state_path)

assert repo_registry.version == REGISTRY_VERSION
assert repo_registry.next_pack_id >= PACK_ID_MIN

print(repo_registry)

YgoMasterIdRegistry(version=1, next_pack_id=9000000, pack_ids={})


## 6. Prima allocazione deterministica

In [6]:
project = make_project(
    "z-pack",
    "a-pack",
    "m-pack",
)

registry = YgoMasterIdRegistry()

assigned = ensure_project_pack_ids(
    project,
    registry,
)

# L'allocazione dei NUOVI ID usa missing_keys ordinati.
assert registry.pack_ids == {
    "a-pack": PACK_ID_MIN,
    "m-pack": PACK_ID_MIN + 1,
    "z-pack": PACK_ID_MIN + 2,
}

# Il risultato restituito segue invece l'ordine dei pack nel ProjectDefinition.
assert list(assigned) == [
    "z-pack",
    "a-pack",
    "m-pack",
]

assert assigned["a-pack"] == PACK_ID_MIN
assert assigned["m-pack"] == PACK_ID_MIN + 1
assert assigned["z-pack"] == PACK_ID_MIN + 2

assert registry.next_pack_id == PACK_ID_MIN + 3

print(assigned)

{'z-pack': 9000002, 'a-pack': 9000000, 'm-pack': 9000001}


## 7. Gli ID esistenti restano stabili

In [7]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MIN + 2,
    pack_ids={
        "pack-a": PACK_ID_MIN,
        "pack-b": PACK_ID_MIN + 1,
    },
)

project = make_project(
    "pack-b",
    "pack-a",
    "pack-c",
)

assigned = ensure_project_pack_ids(
    project,
    registry,
)

assert assigned["pack-a"] == PACK_ID_MIN
assert assigned["pack-b"] == PACK_ID_MIN + 1
assert assigned["pack-c"] == PACK_ID_MIN + 2

assert registry.next_pack_id == PACK_ID_MIN + 3

print("Existing IDs preserved")

Existing IDs preserved


## 8. Un pack rimosso non libera il suo ID

In [8]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MIN + 3,
    pack_ids={
        "old-pack": PACK_ID_MIN,
        "current-pack": PACK_ID_MIN + 1,
        "another-old-pack": PACK_ID_MIN + 2,
    },
)

project = make_project(
    "current-pack",
    "new-pack",
)

assigned = ensure_project_pack_ids(
    project,
    registry,
)

assert assigned["current-pack"] == PACK_ID_MIN + 1
assert assigned["new-pack"] == PACK_ID_MIN + 3

# Le entry non più presenti nel progetto rimangono nel registry.
assert registry.pack_ids["old-pack"] == PACK_ID_MIN
assert registry.pack_ids["another-old-pack"] == PACK_ID_MIN + 2

print("IDs are not recycled")

IDs are not recycled


## 9. Pack reintrodotto → recupera il vecchio ID

In [9]:
project = make_project(
    "old-pack",
    "current-pack",
)

assigned = ensure_project_pack_ids(
    project,
    registry,
)

assert assigned["old-pack"] == PACK_ID_MIN
assert assigned["current-pack"] == PACK_ID_MIN + 1

print("Reintroduced pack restored its stable ID")

Reintroduced pack restored its stable ID


## 10. `next_pack_id` occupato → viene saltato

In [10]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MIN,
    pack_ids={
        "existing": PACK_ID_MIN,
    },
)

project = make_project("new")

assigned = ensure_project_pack_ids(
    project,
    registry,
)

assert assigned["new"] == PACK_ID_MIN + 1
assert registry.next_pack_id == PACK_ID_MIN + 2

print("Occupied IDs skipped correctly")

Occupied IDs skipped correctly


## 11. Save/load round-trip

In [11]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MIN + 2,
    pack_ids={
        "z-pack": PACK_ID_MIN + 1,
        "a-pack": PACK_ID_MIN,
    },
)

with TemporaryDirectory() as tmp:
    path = Path(tmp) / "nested" / "ids.json"

    save_id_registry(
        registry,
        path,
    )

    assert path.exists()
    assert not path.with_suffix(
        path.suffix + ".tmp"
    ).exists()

    loaded = load_id_registry(path)

    assert loaded == registry

    raw = path.read_text(
        encoding="utf-8"
    )

    # Il salvataggio ordina le pack key.
    assert raw.index('"a-pack"') < raw.index('"z-pack"')

print("Round-trip OK")

Round-trip OK


## 12. Versione registry non supportata

In [12]:
registry = YgoMasterIdRegistry(
    version=REGISTRY_VERSION + 1,
)

with TemporaryDirectory() as tmp:
    path = Path(tmp) / "ids.json"

    try:
        save_id_registry(registry, path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Una registry version non supportata "
            "avrebbe dovuto essere rifiutata."
        )

Expected ValueError: Unsupported ID registry version 2.


## 13. ID duplicati nel registry

In [13]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MIN + 1,
    pack_ids={
        "pack-a": PACK_ID_MIN,
        "pack-b": PACK_ID_MIN,
    },
)

try:
    ensure_project_pack_ids(
        make_project("pack-a"),
        registry,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "ID duplicati nel registry non rifiutati."
    )

Expected ValueError: Duplicate pack IDs found in the registry.


## 14. ID fuori range

In [14]:
registry = YgoMasterIdRegistry(
    pack_ids={
        "bad-pack": PACK_ID_MIN - 1,
    },
)

try:
    ensure_project_pack_ids(
        make_project("bad-pack"),
        registry,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "Pack ID fuori range non rifiutato."
    )

Expected ValueError: Pack 'bad-pack' has invalid YgoMaster ID 8999999.


## 15. `next_pack_id` fuori range

In [15]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MAX + 2,
)

try:
    ensure_project_pack_ids(
        make_project("pack"),
        registry,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "next_pack_id fuori range non rifiutato."
    )

Expected ValueError: Invalid next_pack_id: 10000001.


## 16. Pack key duplicate nel ProjectDefinition

In [16]:
project = make_project(
    "same-pack",
    "same-pack",
)

registry = YgoMasterIdRegistry()

try:
    ensure_project_pack_ids(
        project,
        registry,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "Pack key duplicate nel progetto non rifiutata."
    )

Expected ValueError: Project contains duplicate pack keys.


## 17. Esaurimento del range custom

In [17]:
registry = YgoMasterIdRegistry(
    next_pack_id=PACK_ID_MAX + 1,
)

project = make_project(
    "new-pack",
)

try:
    ensure_project_pack_ids(
        project,
        registry,
    )
except ValueError as exc:
    print("Expected ValueError:", exc)
else:
    raise AssertionError(
        "Range esaurito non rilevato."
    )

Expected ValueError: YgoMaster custom pack ID range exhausted.


## 18. Derivazione dello Shop ID

In [18]:
assert get_pack_shop_id(
    PACK_ID_MIN
) == PACK_SHOP_BASE_ID + PACK_ID_MIN

assert get_pack_shop_id(
    PACK_ID_MAX
) == PACK_SHOP_BASE_ID + PACK_ID_MAX

print(
    "First custom shop ID:",
    get_pack_shop_id(PACK_ID_MIN),
)

print(
    "Last custom shop ID:",
    get_pack_shop_id(PACK_ID_MAX),
)

First custom shop ID: 19000000
Last custom shop ID: 19999999


## 19. Shop ID: pack ID fuori range

In [19]:
for invalid_id in [
    PACK_ID_MIN - 1,
    PACK_ID_MAX + 1,
]:
    try:
        get_pack_shop_id(invalid_id)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            f"Pack ID {invalid_id} avrebbe "
            "dovuto essere rifiutato."
        )

Expected ValueError: Invalid custom pack ID: 8999999.
Expected ValueError: Invalid custom pack ID: 10000000.


## 20. JSON root non object

In [20]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "ids.json"

    path.write_text(
        "[]",
        encoding="utf-8",
    )

    try:
        load_id_registry(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "Registry JSON root non object non rifiutata."
        )

Expected ValueError: ID registry root must be an object.


## 21. `pack_ids` non object

In [21]:
with TemporaryDirectory() as tmp:
    path = Path(tmp) / "ids.json"

    path.write_text(
        '''
{
  "version": 1,
  "next_pack_id": 9000000,
  "pack_ids": []
}
'''.strip(),
        encoding="utf-8",
    )

    try:
        load_id_registry(path)
    except ValueError as exc:
        print("Expected ValueError:", exc)
    else:
        raise AssertionError(
            "pack_ids non object non rifiutato."
        )

Expected ValueError: 'pack_ids' must be an object.
